In [4]:
import pandas as pd
import os
import csv
import json # Import json for loading dataset.json

# --- Configuration ---
# --- Pandas Display Options (Add these lines) ---
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 5000)      # Set a wider display width to prevent truncation
pd.set_option('display.max_rows', None)   # Display all rows (if matching_matches has many rows)
# pd.set_option('display.colheader_justify', 'left') # Optional: Adjust column header alignment

# Set the directory for your main Premier League data
DATA_DIR = 'english-premier-league/'
# Set the directory for your individual match stat files (e.g., sample_1_table.csv)
SAMPLE_DATA_DIR = 'training_data/'
# Set the directory for your dataset.json file
DATASET_JSON_PATH = 'dataset.json' # Define path for dataset.json

# List of CSV files to process for the main historical data
CSV_FILES = [f'{year}-{str(int(year) + 1)[-2:]}.csv'
             for year in range(2013, 2024)] # Generates '2013-14.csv', '2014-15.csv', etc. up to '2023-24.csv'

# --- Dynamic TARGET_STATS Population Function ---
def populate_target_stats_from_csv(file_number, data_directory):
    """
    Populates the TARGET_STATS dictionary dynamically from a single CSV file
    with 'Away Team' and 'Home Team' data on separate rows.

    Args:
        file_number (int): The number of the CSV file to read (e.g., 1 for sample_1_table.csv).
        data_directory (str): The directory where the CSV files are located (e.g., SAMPLE_DATA_DIR).

    Returns:
        tuple: A tuple containing the dynamically populated TARGET_STATS dictionary and
               a list of column names extracted from the sample CSV's header (excluding 'Team').
               Returns (None, None) if the file/data is not found or an error occurs.
    """
    file_name = f"sample_{file_number}_table.csv"
    file_path = os.path.join(data_directory, file_name)

    dynamic_target_stats = {}
    sample_columns = []

    try:
        with open(file_path, mode='r', encoding='utf-8') as csvfile: # Added encoding
            reader = csv.reader(csvfile)

            # Read the header row and strip whitespace
            header = [h.strip() for h in next(reader)]

            # Find the indices of the columns we need
            column_indices = {}
            for i, col_name in enumerate(header):
                column_indices[col_name] = i
                if col_name != 'Team': # Collect column names from sample, excluding 'Team'
                    sample_columns.append(col_name)

            if 'Team' not in column_indices:
                print(f"Error: 'Team' column not found in header of {file_path}")
                return None, None

            away_data = None
            home_data = None

            # Read the next two rows (Away Team and Home Team)
            try:
                row1 = [d.strip() for d in next(reader)]
                row2 = [d.strip() for d in next(reader)]
            except StopIteration:
                print(f"Error: Not enough data rows in {file_path} (expected header + 2 data rows).")
                return None, None

            # Determine which row is Away and which is Home
            team_col_idx = column_indices['Team']
            if row1[team_col_idx].lower() == 'away team' and row2[team_col_idx].lower() == 'home team':
                away_data = row1
                home_data = row2
            elif row1[team_col_idx].lower() == 'home team' and row2[team_col_idx].lower() == 'away team':
                away_data = row2
                home_data = row1
            else:
                print(f"Error: Could not identify Home and Away teams in {file_path}. Found: '{row1[team_col_idx]}' and '{row2[team_col_idx]}'")
                return None, None

            # Helper to safely get integer value
            def get_stat_value(data_row, stat_name_in_csv):
                if stat_name_in_csv in column_indices:
                    try:
                        # Handle potential empty strings before converting to int
                        value_str = data_row[column_indices[stat_name_in_csv]]
                        if value_str == '':
                            return None # Or a default like 0, depending on requirements
                        return int(value_str)
                    except ValueError:
                        # print(f"Warning: Could not convert value for '{stat_name_in_csv}' to int in {file_path}.")
                        return None # Or handle as error
                return None

            # Populate dynamic_target_stats for Away Team
            goals_away = get_stat_value(away_data, 'Goals')
            if goals_away is not None: dynamic_target_stats['FTAG'] = {'operator': '==', 'value': goals_away}
            shots_away = get_stat_value(away_data, 'Shots')
            if shots_away is not None: dynamic_target_stats['AS'] = {'operator': '==', 'value': shots_away}
            fouls_away = get_stat_value(away_data, 'Fouls')
            if fouls_away is not None: dynamic_target_stats['AF'] = {'operator': '==', 'value': fouls_away}
            yellow_away = get_stat_value(away_data, 'Yellow Cards')
            if yellow_away is not None: dynamic_target_stats['AY'] = {'operator': '==', 'value': yellow_away}
            red_away = get_stat_value(away_data, 'Red Cards')
            if red_away is not None: dynamic_target_stats['AR'] = {'operator': '==', 'value': red_away}
            corners_away = get_stat_value(away_data, 'Corner Kicks')
            if corners_away is not None: dynamic_target_stats['AC'] = {'operator': '==', 'value': corners_away}

            # Populate dynamic_target_stats for Home Team
            goals_home = get_stat_value(home_data, 'Goals')
            if goals_home is not None: dynamic_target_stats['FTHG'] = {'operator': '==', 'value': goals_home}
            shots_home = get_stat_value(home_data, 'Shots')
            if shots_home is not None: dynamic_target_stats['HS'] = {'operator': '==', 'value': shots_home}
            fouls_home = get_stat_value(home_data, 'Fouls')
            if fouls_home is not None: dynamic_target_stats['HF'] = {'operator': '==', 'value': fouls_home}
            yellow_home = get_stat_value(home_data, 'Yellow Cards')
            if yellow_home is not None: dynamic_target_stats['HY'] = {'operator': '==', 'value': yellow_home}
            red_home = get_stat_value(home_data, 'Red Cards')
            if red_home is not None: dynamic_target_stats['HR'] = {'operator': '==', 'value': red_home}
            corners_home = get_stat_value(home_data, 'Corner Kicks')
            if corners_home is not None: dynamic_target_stats['HC'] = {'operator': '==', 'value': corners_home}

            return dynamic_target_stats, sample_columns

    except FileNotFoundError:
        print(f"Error: Sample file not found at {file_path}")
        return None, None
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, None

## Core Data Loading and Filtering Functions
def load_and_combine_data(data_directory, csv_files, columns_to_keep=None):
    """
    Loads multiple CSV files from a directory into a single Pandas DataFrame.
    Assumes CSVs contain Premier League match data.
    Optionally drops columns not in `columns_to_keep`.
    """
    all_data = []
    for filename in csv_files:
        filepath = os.path.join(data_directory, filename)
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath, encoding='latin1') # Keep latin1 if that's the known encoding
                df['Season'] = filename.split('.')[0] # Add a 'Season' column

                if columns_to_keep:
                    # Ensure 'Season' is not accidentally dropped if columns_to_keep is used
                    effective_columns_to_keep = set(columns_to_keep)
                    effective_columns_to_keep.add('Season')

                    # Identify columns to drop: those in df but not in effective_columns_to_keep
                    cols_to_drop = [col for col in df.columns if col not in effective_columns_to_keep]
                    if cols_to_drop:
                        df = df.drop(columns=cols_to_drop)

                all_data.append(df)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        else:
            print(f"File not found: {filename}")

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data loaded. Please check your DATA_DIR and CSV_FILES list.")
        return pd.DataFrame()

def find_matches_by_stats(dataframe, target_stats):
    """
    Filters a DataFrame to find matches that meet the specified statistical criteria.
    """
    if dataframe.empty:
        return pd.DataFrame()

    condition = pd.Series(True, index=dataframe.index)

    for col, criteria in target_stats.items():
        if col in dataframe.columns:
            operator = criteria['operator']
            value = criteria['value']

            # Ensure the column's data type is compatible for comparison
            # Attempt to convert to numeric if not already, ignoring errors
            if not pd.api.types.is_numeric_dtype(dataframe[col]):
                try:
                    # Attempt to convert, coercing errors will turn unconvertible values to NaT/NaN
                    dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce')
                except TypeError: # Should be rare with errors='coerce'
                    print(f"Warning: Cannot convert column '{col}' to numeric for comparison.")
                    continue # Skip this column if conversion fails

            # Apply condition only if value is not None (from get_stat_value) and column is not NaN
            if value is not None: # Value from target_stats is a valid number
                # Create a boolean series for the current condition, handling NaNs in the DataFrame column
                # NaNs in the dataframe column should not satisfy the condition unless explicitly handled
                if operator == '>':
                    condition &= dataframe[col].gt(value) & dataframe[col].notna()
                elif operator == '<':
                    condition &= dataframe[col].lt(value) & dataframe[col].notna()
                elif operator == '==':
                    condition &= dataframe[col].eq(value) & dataframe[col].notna()
                elif operator == '>=':
                    condition &= dataframe[col].ge(value) & dataframe[col].notna()
                elif operator == '<=':
                    condition &= dataframe[col].le(value) & dataframe[col].notna()
                elif operator == '!=':
                    condition &= dataframe[col].ne(value) & dataframe[col].notna() # Or handle NaNs as not equal too
                else:
                    print(f"Warning: Unknown operator '{operator}' for column '{col}'.")
        else:
            # print(f"Warning: Column '{col}' from target_stats not found in historical DataFrame.")
            pass # Silently ignore if column doesn't exist in main data, or print warning

    return dataframe[condition]

# --- Main Execution Logic for Automation ---
if __name__ == "__main__":
    print(f"Using main data directory: {DATA_DIR}")
    print(f"Using sample data directory: {SAMPLE_DATA_DIR}")
    print(f"CSV files to process (main data): {CSV_FILES}")

    # Create a set of columns expected in the main dataframe, derived from the sample's relevant columns
    # and common match identifiers.
    main_df_relevant_columns = set([
        'HomeTeam', 'AwayTeam', 'FTR', 'Div', 'Date', 'Referee', # Common match identifiers
        'FTHG', 'FTAG', 'HS', 'AS', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'HC', 'AC', 'HTHG', 'HTAG', 'HTR',
        # Include all relevant stat columns for display (some might be redundant with abbreviations)
        # 'Shots', 'ShotsOnTarget', 'Fouls', 'Corners', 'YellowCards', 'RedCards', # These are generic names from sample
        'HST', 'AST', # 'B365H', 'B365D', 'B365A' # Example betting odds, can add more if present
    ])
    # Add abbreviated stat columns explicitly if they are distinct and needed
    stat_abbrs = ['FTHG', 'FTAG', 'HS', 'AS', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'HC', 'AC', 'HST', 'AST']
    for stat in stat_abbrs:
        main_df_relevant_columns.add(stat)


    print("\nLoading and combining main Premier League data...")
    combined_football_data = load_and_combine_data(DATA_DIR, CSV_FILES, columns_to_keep=main_df_relevant_columns)

    if combined_football_data.empty:
        print("Failed to load any main Premier League data. Exiting.")
    else:
        print(f"\nSuccessfully loaded {len(combined_football_data)} rows of historical Premier League data.")
        print("First 5 rows of combined main data:")
        try:
            from IPython.display import display
            display(combined_football_data.head())
        except ImportError:
            print(combined_football_data.head().to_string())

        sample_files = [f for f in os.listdir(SAMPLE_DATA_DIR) if f.startswith('sample_') and f.endswith('_table.csv')]
        sample_numbers = []
        for f_name in sample_files:
            try:
                num_part = f_name.replace('sample_', '').replace('_table.csv', '')
                sample_numbers.append(int(num_part))
            except ValueError:
                print(f"Could not parse sample number from filename: {f_name}")
                continue
        sample_numbers.sort()

        print(f"\nFound sample numbers to process: {sample_numbers}")
        print("\n--- Matching Sample Files to Historical Data ---")
        results = []
        total_samples_processed = 0
        total_matches_found_exactly_one = 0 # Renamed for clarity
        total_multiple_matches_found = 0
        total_no_matches_found = 0 # For samples where dynamic_target_stats was valid but no match
        total_sample_parse_errors = 0 # For samples that couldn't be parsed

        display_cols_for_match = [
            'Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTR', 'FTHG', 'FTAG',
            'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
        ]
        # Ensure all display_cols_for_match are valid columns in combined_football_data if possible
        # (though find_matches_by_stats handles missing columns)

        for sample_num in sample_numbers:
            total_samples_processed += 1
            print(f"\nProcessing Sample # {sample_num}...")
            dynamic_target_stats, _ = populate_target_stats_from_csv(sample_num, SAMPLE_DATA_DIR)

            mapping_status = "No Conclusive Result" # Default status
            match_details_string = "N/A"

            if dynamic_target_stats:
                if not dynamic_target_stats: # Empty dict means no valid stats were parsed
                    print(f"  Sample # {sample_num}: No valid stats parsed from sample file.")
                    mapping_status = "Sample Parsed (No Stats)"
                    match_details_string = "N/A (No stats to match)"
                    total_sample_parse_errors +=1 # Or a more specific counter
                else:
                    print(f"  Sample # {sample_num}: Target stats: {dynamic_target_stats}")
                    matching_matches = find_matches_by_stats(combined_football_data.copy(), dynamic_target_stats) # Use a copy

                    if not matching_matches.empty:
                        if len(matching_matches) == 1:
                            total_matches_found_exactly_one += 1
                            match_info = matching_matches.iloc[0]
                            mapping_status = (
                                f"{match_info.get('HomeTeam', 'N/A')} vs {match_info.get('AwayTeam', 'N/A')} on {match_info.get('Date', 'N/A')} "
                                f"(Season: {match_info.get('Season', 'N/A')})"
                            )
                            match_details_list = []
                            for col in display_cols_for_match:
                                if col in match_info and pd.notna(match_info[col]):
                                    match_details_list.append(f"{col}: {match_info[col]}")
                                else:
                                    match_details_list.append(f"{col}: N/A")
                            match_details_string = "; ".join(match_details_list)
                            print(f"  Sample # {sample_num}: Found 1 match: {mapping_status}")
                        else:
                            total_multiple_matches_found +=1
                            mapping_status = f"Multiple Matches Found ({len(matching_matches)}): "
                            first_matches_info_list = []
                            for i, (_, match_info_row) in enumerate(matching_matches.head(3).iterrows()):
                                first_matches_info_list.append(
                                    f"{match_info_row.get('HomeTeam','N/A')} vs {match_info_row.get('AwayTeam','N/A')} on {match_info_row.get('Date','N/A')} (S: {match_info_row.get('Season','N/A')})"
                                )
                            mapping_status += "; ".join(first_matches_info_list)
                            if len(matching_matches) > 3:
                                mapping_status += " and more..."
                            match_details_string = "N/A (Multiple Matches)"
                            print(f"  Sample # {sample_num}: Found {len(matching_matches)} matches.")
                    else:
                        total_no_matches_found += 1
                        mapping_status = "No Matches Found"
                        match_details_string = "N/A (No historical matches met criteria)"
                        print(f"  Sample # {sample_num}: No matches found for the given stats.")
            else:
                total_sample_parse_errors += 1
                mapping_status = "Sample Parsing Error"
                match_details_string = "N/A (Sample parsing error or file not found/empty)"
                print(f"  Sample # {sample_num}: Error parsing sample file or file was empty/not found.")


            results.append({
                'Sample #': sample_num,
                'Mapping': mapping_status,
                'Match Stats (Historical Data)': match_details_string
            })

        results_df = pd.DataFrame(results)

        print("\n--- Automated Match Mapping Results ---")
        try:
            from IPython.display import display
            display(results_df)
        except ImportError:
            print(results_df.to_string())

        print(f"\n--- Summary ---")
        print(f"Total samples processed: {total_samples_processed}")
        print(f"Total successfully parsed samples with exactly one match: {total_matches_found_exactly_one}")
        print(f"Total successfully parsed samples with multiple matches: {total_multiple_matches_found}")
        print(f"Total successfully parsed samples with no matches found: {total_no_matches_found}")
        print(f"Total samples with parsing errors/no stats: {total_sample_parse_errors}")


        # --- Generating final_df with dataset.json mapping (Scenario B) ---
        print("\n--- Generating final_df with dataset.json mapping (Scenario B) ---")
        dataset_list_from_json = []
        try:
            with open(DATASET_JSON_PATH, 'r', encoding='utf-8') as file: # Added encoding
                dataset_list_from_json = json.load(file)
            print(f"Loaded {len(dataset_list_from_json)} entries from {DATASET_JSON_PATH}.")
        except FileNotFoundError:
            print(f"Error: {DATASET_JSON_PATH} not found. Cannot perform ID mapping for final_df.")
        except json.JSONDecodeError:
            print(f"Error: Could not decode JSON from {DATASET_JSON_PATH}.")
        except Exception as e:
            print(f"An unexpected error occurred while loading {DATASET_JSON_PATH}: {e}")

        # Filter the results for rows that represent a successful, unique match or multiple matches.
        # You might want to adjust this filter based on what you consider a "valid" mapping for final_df.
        # For example, only include those with exactly one match:
        # filtered_results_for_final_df = [row for row in results if "vs" in row['Mapping'] and not "Multiple Matches" in row['Mapping']]
        # Or, as per your original code, any row not "No Conclusive Result" (which is broad)
        # The original logic was: `row['Mapping'] != "No Conclusive Result"`
        # "No Conclusive Result" was the default, changed if parsing error, no stats, no match, 1 match, or multiple.
        # Let's stick to a similar idea: include if we have some form of match info or even multiple matches.
        # Exclude "Sample Parsing Error", "No Matches Found", "Sample Parsed (No Stats)".
        valid_mapping_keywords = ["vs", "Multiple Matches Found"] # Keywords indicating a match was found or multiple were
        
        # The original filtered_results selected rows where Mapping was NOT "No Conclusive Result".
        # "No Conclusive Result" was the initial state for a sample.
        # It would be overwritten by:
        #   - "Sample Parsed (No Stats)"
        #   - "Found 1 match..." (contains "vs")
        #   - "Multiple Matches Found..."
        #   - "No Matches Found"
        #   - "Sample Parsing Error"
        # So, `row['Mapping'] != "No Conclusive Result"` means any processed sample.
        # The user's code used this:
        # filtered_results = [row for row in results if row['Mapping'] != "No Conclusive Result"]
        # This will include "No Matches Found" and "Sample Parsed (No Stats)".
        # If you only want rows where a match (single or multiple) was identified:
        
        # Using the original filter criteria from the user's provided script:
        selected_results_for_id_mapping = [row for row in results if row['Mapping'] != "No Conclusive Result"]
        print(f"Filtered results (Mapping not 'No Conclusive Result'): {len(selected_results_for_id_mapping)} rows for ID mapping.")


        # Get the first 275 rows from these filtered results
        # The user's original code had:
        # selected_results = filtered_results[:400]
        # This implies taking the top 275 from the `results` list that met the criteria.
        final_df_source_rows = selected_results_for_id_mapping[:400]
        print(f"Selected first {len(final_df_source_rows)} (up to 275) filtered results for final_df.")

        data_for_final_df = []
        if dataset_list_from_json: # Proceed only if dataset was loaded successfully
            for result_item in final_df_source_rows:
                sample_num = result_item.get('Sample #')

                if sample_num is not None:
                    # Scenario B: sample_num (1-based) maps to 0-based list index
                    target_json_index = sample_num - 1

                    if 0 <= target_json_index < len(dataset_list_from_json):
                        json_record_entry = dataset_list_from_json[target_json_index]
                        id_value_from_record = json_record_entry.get('id') # Safely get 'id'

                        if id_value_from_record is not None:
                            df_row_data = result_item.copy()
                            df_row_data['ID'] = id_value_from_record
                            data_for_final_df.append(df_row_data)
                        else:
                            print(f"Warning: 'id' field missing in dataset.json entry at index {target_json_index} (for Sample # {sample_num}).")
                    else:
                        print(f"Warning: Sample # {sample_num} (maps to index {target_json_index}) is out of range for dataset.json (size {len(dataset_list_from_json)}).")
                else:
                    print(f"Warning: 'Sample #' is missing or None in a result item: {result_item}")
        else:
            print(f"Skipping generation of final_df because {DATASET_JSON_PATH} was not loaded or is empty.")

        final_df = pd.DataFrame(data_for_final_df)

        print("\n--- Head of final_df ---")
        if not final_df.empty:
            try:
                from IPython.display import display
                display(final_df.head())
            except ImportError:
                print(final_df.head().to_string())
        else:
            print("final_df is empty.")

        print(f"\nFinal DataFrame 'final_df' has {len(final_df)} rows.")

Using main data directory: english-premier-league/
Using sample data directory: training_data/
CSV files to process (main data): ['2013-14.csv', '2014-15.csv', '2015-16.csv', '2016-17.csv', '2017-18.csv', '2018-19.csv', '2019-20.csv', '2020-21.csv', '2021-22.csv', '2022-23.csv', '2023-24.csv']

Loading and combining main Premier League data...

Successfully loaded 3840 rows of historical Premier League data.
First 5 rows of combined main data:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,E0,17/08/13,Arsenal,Aston Villa,1,3,A,1,1,D,A Taylor,16,9,4,4,15,18,4,3,4,5,1,0,2013-14
1,E0,17/08/13,Liverpool,Stoke,1,0,H,1,0,H,M Atkinson,26,10,11,4,11,11,12,6,1,1,0,0,2013-14
2,E0,17/08/13,Norwich,Everton,2,2,D,0,0,D,M Oliver,8,19,2,6,13,10,6,8,2,0,0,0,2013-14
3,E0,17/08/13,Sunderland,Fulham,0,1,A,0,0,D,N Swarbrick,20,5,3,1,14,14,6,1,0,3,0,0,2013-14
4,E0,17/08/13,Swansea,Man United,1,4,A,0,2,A,P Dowd,17,15,6,7,13,10,7,4,1,3,0,0,2013-14



Found sample numbers to process: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215

,Sample #,Mapping,Match Stats (Historical Data)
0,1,No Matches Found,N/A (No historical matches met criteria)
1,2,Man United vs Tottenham on 01/01/14 (Season: 2...,Season: 2013-14; Date: 01/01/14; HomeTeam: Man...
2,3,West Brom vs Newcastle on 01/01/14 (Season: 20...,Season: 2013-14; Date: 01/01/14; HomeTeam: Wes...
3,4,Liverpool vs Hull on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Liv...
4,5,Stoke vs Everton on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Sto...
5,6,Swansea vs Man City on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Swa...
6,7,Fulham vs West Ham on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Ful...
7,8,No Matches Found,N/A (No historical matches met criteria)
8,9,Southampton vs West Brom on 11/01/14 (Season: ...,Season: 2013-14; Date: 11/01/14; HomeTeam: Sou...
9,10,Hull vs Chelsea on 11/01/14 (Season: 2013-14),Season: 2013-14; Date: 11/01/14; HomeTeam: Hul...



--- Summary ---
Total samples processed: 3771
Total successfully parsed samples with exactly one match: 2718
Total successfully parsed samples with multiple matches: 0
Total successfully parsed samples with no matches found: 1053
Total samples with parsing errors/no stats: 0

--- Generating final_df with dataset.json mapping (Scenario B) ---
Loaded 3771 entries from dataset.json.
Filtered results (Mapping not 'No Conclusive Result'): 3771 rows for ID mapping.
Selected first 400 (up to 275) filtered results for final_df.

--- Head of final_df ---


,Sample #,Mapping,Match Stats (Historical Data),ID
0,1,No Matches Found,N/A (No historical matches met criteria),25513261
1,2,Man United vs Tottenham on 01/01/14 (Season: 2...,Season: 2013-14; Date: 01/01/14; HomeTeam: Man...,25513267
2,3,West Brom vs Newcastle on 01/01/14 (Season: 20...,Season: 2013-14; Date: 01/01/14; HomeTeam: Wes...,25513268
3,4,Liverpool vs Hull on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Liv...,25513270
4,5,Stoke vs Everton on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Sto...,25513272



Final DataFrame 'final_df' has 400 rows.
